# MorphoNet-BT — 05: Complexity and Efficiency
## Objective and definitions
Measure the frozen original architecture per float32 image `[1,1,256,256]`, without training/backward/baselines. Separate model(image) from model(image, return_aux=True), both in eval. The latter measures auxiliary forward cost, not full training/backward. Primary MACs count Conv2d/Linear with 1 MAC=2 FLOPs; other operations are reported separately and in an expanded estimate. Default THOP combines MACs with BN/pooling counts; its mixed total must not be indiscriminately doubled.

## Public-copy execution policy
This is an English rendering of a historical research notebook. Outputs and execution counts were cleared; no cell was rerun for Phase 18. Historical findings below describe earlier runs only. Every code cell is gated, including setup cells that may perform model checks, preprocessing or writes. Enable `CONFIRM_PUBLIC_WORKFLOW` deliberately in the first code cell only after reviewing [the reproduction guide](../docs/reproducibility.md). Existing stage-specific confirmations must also be enabled separately. Run from the repository root or set `MORPHONET_ROOT` to a private writable experiment workspace with the same source/config layout; set `MORPHONET_DATA_ROOT` for dataset discovery. Original source/configuration fingerprints, private split manifests and checkpoints are not portable artifacts; do not bypass their validation. See the guide for prerequisites and limitations.

**Model identity:** this phase concerns the original full model, later A5. The principal manuscript A4 uses `build_ablation("A4")` and fixed mean fusion, not AMGF.


In [ ]:
CONFIRM_PUBLIC_WORKFLOW = False  # Explicit opt-in; no cell has been rerun.
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
from pathlib import Path
import sys, json, hashlib, platform, tempfile
from importlib.metadata import version
PROJECT_ROOT=Path(__import__("os").environ.get("MORPHONET_ROOT", str(Path.cwd()))).resolve()
if PROJECT_ROOT.name == "notebooks": PROJECT_ROOT=PROJECT_ROOT.parent
sys.path.insert(0,str(PROJECT_ROOT))
assert Path(sys.prefix).name.lower()=="dl"
import numpy as np
import pandas as pd
import torch
import yaml
from IPython.display import display
from src.models.morphonet_bt import MorphoNetBT, MODEL_CONFIG
from src.data.dataset import verify_frozen_splits
from src.utils.complexity import (parameter_breakdown, fingerprint, thop_measure, trace_layers,
    additional_operations, depthwise_audit, cuda_memory, cuda_latency)
OUTPUT=PROJECT_ROOT/"results/complexity"
OUTPUT.mkdir(parents=True,exist_ok=True)
torch.manual_seed(42)
torch.backends.cudnn.benchmark=False
torch.backends.cudnn.deterministic=True
torch.backends.cuda.matmul.allow_tf32=False
torch.backends.cudnn.allow_tf32=False
config_path=PROJECT_ROOT/"configs/default.yaml"
config=yaml.safe_load(config_path.read_text(encoding="utf-8"))
config_hash=hashlib.sha256(config_path.read_bytes()).hexdigest()
assert config["model"]==MODEL_CONFIG
model=MorphoNetBT(**config["model"]).eval()
x=torch.randn(1,1,256,256,dtype=torch.float32)
print("PyTorch:",torch.__version__,"THOP:",version("thop"),"CUDA:",torch.cuda.is_available())

## Frozen architecture/configuration checks and fingerprint

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
phase4=json.loads((PROJECT_ROOT/"results/architecture/validation_results.json").read_text())
phase4_hashes=json.loads((PROJECT_ROOT/"results/architecture/frozen_input_hashes.json").read_text())
assert verify_frozen_splits(PROJECT_ROOT)==phase4_hashes["splits"]
for name,digest in phase4_hashes["preprocessing_sources"].items():
    assert hashlib.sha256((PROJECT_ROOT/name).read_bytes()).hexdigest()==digest
approved=yaml.safe_load((PROJECT_ROOT/"results/architecture/preprocessing_v1_approved.yaml").read_text())
assert {k:v for k,v in config.items() if k!="model"}==approved
before=fingerprint(model,config["model"],PROJECT_ROOT)
assert before["parameters"]==phase4["parameter_count_basic"]==579599
fingerprint_path=OUTPUT/"architecture_fingerprint.json"
if fingerprint_path.exists():
    assert json.loads(fingerprint_path.read_text())["fingerprint_sha256"]==before["fingerprint_sha256"]
fingerprint_path.write_text(json.dumps(before,indent=2),encoding="utf-8")
print("Architecture fingerprint:",before["fingerprint_sha256"])
# Phase 4 did not save model-source hashes: this is the first source fingerprint baseline.
print("PASS: config/count match Phase 4; source hashes recorded before profiling")

## Parameter counts and allocation
PyTorch counts are independent of the profiler. Nontrainable parameters exclude BatchNorm running-statistic buffers, which are included separately in storage. Components are nonoverlapping: BAA is separated from Local; region/distance predictors from shared morphology/boundary guidance.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
total=sum(p.numel() for p in model.parameters())
trainable=sum(p.numel() for p in model.parameters() if p.requires_grad)
nontrainable=sum(p.numel() for p in model.parameters() if not p.requires_grad)
parameters=parameter_breakdown(model)
assert int(parameters.parameters.sum())==total==579599
parameters.to_csv(OUTPUT/"parameter_breakdown.csv",index=False)
display(pd.Series({"total":total,"trainable":trainable,"nontrainable":nontrainable,"millions":total/1e6}))
display(parameters.sort_values("parameters",ascending=False))

## Depthwise, actual execution path and operation audit

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
depthwise=depthwise_audit(model)
depthwise.to_csv(OUTPUT/"depthwise_audit.csv",index=False)
display(depthwise)
layer_tables={}
for path,aux in [("inference",False),("auxiliary",True)]:
    layer_tables[path]=trace_layers(model,x,aux)
    layer_tables[path].to_csv(OUTPUT/f"{path}_layer_operations.csv",index=False)
    called=set(layer_tables[path].layer)
    assert "morphology_head.boundary_predictor" in called
    assert "morphology_head.features.0.0" in called
    for head in ("region_predictor","distance_predictor"):
        assert (f"morphology_head.{head}" in called)==aux
    expected={"stem.project.0":[1,32,128,128],"stage1.1.project.0":[1,48,64,64],
              "stage2.1.project.0":[1,80,32,32],"stage3.2.project.0":[1,128,16,16],
              "stage4.1.project.0":[1,192,8,8]}
    for name,shape in expected.items():
        assert json.loads(layer_tables[path].set_index("layer").loc[name,"output_shape"])==shape
display(layer_tables["inference"][["layer","input_shape","output_shape","conv_linear_macs"]])
with torch.inference_mode():
    logits=model(x)
    aux_outputs=model(x,return_aux=True)
assert logits.shape==(1,3) and torch.allclose(logits,aux_outputs["logits"],atol=1e-6,rtol=1e-5)
del logits,aux_outputs
print("PASS: boundary/shared head runs in inference; region/distance heads are skipped")

## Boundary-guidance audit
Stage 2 at 32×32 → bottleneck 80→32 plus depthwise 3×3 → boundary predictor → bilinear to 16×16 → local-score addition → sigmoid → BAA → local projection → AMGF → classification. No ground-truth mask or region/distance decoding at inference. These outputs are already skipped. No convolution operates at 256×256; auxiliary mode alone produces three bilinear maps and distance sigmoid at this resolution.

## THOP, independent counts and additional operations
Compare default THOP with zero BN/pooling rules to isolate Conv/Linear, without architecture changes. Cross-check hooks on every executed Conv2d/Linear. TorchDispatchMode observes functional operations. Expanded estimate: BN=4 operations/element without fusion, GAP=sum/division, bias adds, add/mul/sub, bilinear≈7/element with precomputed interpolation weights, sigmoid≈4 and SiLU≈5 with unit-cost exp. This arithmetic estimate is not a device-instruction count; it excludes memory traffic, interpolation-coordinate calculation, actual exp cost and kernel launches.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
profiles={}
module_tables=[]
for path,aux in [("inference",False),("auxiliary",True)]:
    measured=thop_measure(model,x,aux)
    layers=layer_tables[path]
    independent=int(layers.conv_linear_macs.sum())
    assert independent==measured["thop_conv_linear_macs"], "Major convolution accounting mismatch"
    mixed_expected=independent+int(layers.batchnorm_scalar_ops.sum())+int(layers.thop_pooling_ops.sum())
    assert measured["thop_default_mixed_operations"]==mixed_expected, "Unexplained THOP discrepancy"
    extra,ops=additional_operations(model,x,layers,aux)
    ops.to_csv(OUTPUT/f"{path}_aten_operations.csv",index=False)
    profiles[path]={**measured,"independent_conv_linear_macs":independent,
                    "conv_linear_flops":2*independent,**extra,
                    "expanded_flops_estimate":2*independent+extra["additional_ops_estimate"]}
    grouped=layers.groupby("module",as_index=False).conv_linear_macs.sum()
    grouped=parameters.merge(grouped,on="module",how="left").fillna({"conv_linear_macs":0})
    grouped["path"]=path
    grouped["conv_linear_flops"]=2*grouped.conv_linear_macs
    grouped["percentage_of_total_MACs"]=100*grouped.conv_linear_macs/independent
    module_tables.append(grouped)
module_table=pd.concat(module_tables,ignore_index=True)
module_table.to_csv(OUTPUT/"module_complexity.csv",index=False)
(OUTPUT/"profiling_results.json").write_text(json.dumps(profiles,indent=2),encoding="utf-8")
display(pd.DataFrame(profiles).T)
display(module_table[module_table.path.eq("inference")].sort_values("conv_linear_macs",ascending=False))
assert profiles["inference"]["expanded_flops_estimate"]<1e9, "Inference budget exceeded: stop and report without redesign"
assert profiles["auxiliary"]["independent_conv_linear_macs"]-profiles["inference"]["independent_conv_linear_macs"]==2*32*32*32
print("PASS: THOP matches independent Conv/Linear accounting; expanded inference estimate below 1 GFLOP")

## Representative manual MAC calculation
MACs = B×Hout×Wout×Cout×(Cin/groups)×Kh×Kw, excluding bias.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
manual=[]
for name in ("stage1.0.depthwise.0","stage1.0.project.0"):
    layer=dict(model.named_modules())[name]
    observed=layer_tables["inference"].set_index("layer").loc[name]
    b,c,h,w=json.loads(observed.output_shape)
    value=b*h*w*c*(layer.in_channels//layer.groups)*layer.kernel_size[0]*layer.kernel_size[1]
    assert value==int(observed.conv_linear_macs)
    manual.append({"layer":name,"manual_MACs":value,"observed_MACs":int(observed.conv_linear_macs),"status":"PASS"})
display(pd.DataFrame(manual))
pd.DataFrame(manual).to_csv(OUTPUT/"manual_mac_checks.csv",index=False)

## Parameter storage and untrained state_dict
MB=10^6 bytes; MiB=2^20 bytes. state_dict size includes buffers and serialization, not just raw parameters or runtime memory.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
storage={"fp32_parameter_bytes":total*4,"fp32_MB":total*4/1e6,"fp32_MiB":total*4/2**20,
         "fp16_parameter_bytes_theoretical":total*2,"fp16_MB_theoretical":total*2/1e6,
         "buffer_bytes":sum(b.numel()*b.element_size() for b in model.buffers())}
with tempfile.TemporaryDirectory(dir=OUTPUT) as temp:
    temp_path=Path(temp)/"untrained_state_dict.pt"
    torch.save(model.state_dict(),temp_path)
    storage["serialized_state_dict_bytes"]=temp_path.stat().st_size
storage["serialized_state_dict_MB"]=storage["serialized_state_dict_bytes"]/1e6
storage["serialized_state_dict_MiB"]=storage["serialized_state_dict_bytes"]/2**20
display(pd.Series(storage))
print("Temporary untrained state_dict removed; no checkpoint retained")

## Forward-only GPU memory
Measure batches 1/4 when memory permits, with inference_mode for both paths. Baseline includes model, buffers and input; peak increment estimates activations/temporaries/forward outputs. PyTorch allocator memory is not total process/CUDA-context memory and excludes backward/optimizer. Report reserved separately from allocated.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
assert torch.cuda.is_available(), "CUDA unavailable: report GPU measurements as unavailable"
model=model.to("cuda").eval()
free,total_vram=torch.cuda.mem_get_info()
assert free>512*2**20, "Insufficient free VRAM: stop memory test"
memory_rows=[]
for batch_size in (1,4):
    for aux in (False,True):
        row=cuda_memory(model,batch_size,aux)
        assert row["peak_allocated_bytes"]<total_vram*.8, "Abnormal GPU memory usage"
        memory_rows.append(row)
memory=pd.DataFrame(memory_rows)
for column in ("baseline_allocated_bytes","peak_allocated_bytes","incremental_forward_peak_bytes","peak_reserved_bytes"):
    memory[column.replace("_bytes","_MiB")]=memory[column]/2**20
memory.to_csv(OUTPUT/"memory_results.csv",index=False)
display(memory)

## RTX 3070 Laptop GPU latency
Batch 1, FP32, eval/inference_mode, without AMP/TF32/compile. Use 30 warmups and 100 timed repetitions per path, CUDA-synchronized before/after each. Wall-clock includes kernel launch/wait, excludes MRI loading/preprocessing/CPU-to-GPU transfer. images/sec=1000/mean_ms for sequential single images, not large-batch throughput. Results depend on hardware, thermal/power state and system load; they are not universal model speed.

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
latency_rows=[]
raw_timings=[]
for aux in (False,True):
    row,samples=cuda_latency(model,aux,warmup=30,repeats=100)
    latency_rows.append(row)
    raw_timings.extend({"path":row["path"],"iteration":i+1,"ms":value} for i,value in enumerate(samples))
latency=pd.DataFrame(latency_rows)
latency.to_csv(OUTPUT/"latency_results.csv",index=False)
pd.DataFrame(raw_timings).to_csv(OUTPUT/"latency_raw.csv",index=False)
display(latency)

## Final report and architecture-integrity verification

In [ ]:
if not globals().get("CONFIRM_PUBLIC_WORKFLOW", False):
    raise RuntimeError("Execution disabled. Read docs/reproducibility.md and explicitly enable this workflow.")
after=fingerprint(model,config["model"],PROJECT_ROOT)
assert after==before
assert hashlib.sha256(config_path.read_bytes()).hexdigest()==config_hash
assert verify_frozen_splits(PROJECT_ROOT)==phase4_hashes["splits"]
for name,digest in phase4_hashes["preprocessing_sources"].items():
    assert hashlib.sha256((PROJECT_ROOT/name).read_bytes()).hexdigest()==digest
summary={"input_shape":[1,1,256,256],"dtype":"float32","total_parameters":total,
         "trainable_parameters":trainable,"nontrainable_parameters":nontrainable,
         "storage":storage,"profiles":profiles,"latency":latency_rows,"memory":memory.to_dict("records"),
         "profiler":"THOP "+version("thop"),"torch":torch.__version__,"cuda":torch.version.cuda,
         "numpy":np.__version__,"python":platform.python_version(),"tf32":False,"cudnn_benchmark":False,
         "seed":42,"mac_flop_convention":"Conv/Linear MACs; 1 MAC = 2 FLOPs; additional operations estimated separately",
         "under_1M_parameters":total<1e6,"under_1G_expanded_inference_estimate":profiles["inference"]["expanded_flops_estimate"]<1e9,
         "frozen_inputs_unchanged":True,"training":"NOT RUN","phase6":"NOT STARTED"}
(OUTPUT/"complexity_summary.json").write_text(json.dumps(summary,indent=2),encoding="utf-8")
notes="""Default THOP mixes Conv/Linear MACs, BN (4/element), and adaptive pooling (input plus output elements).
Conv/Linear counts therefore use zero BN/pooling rules and independent counting of all executed layers.
Unregistered containers do not imply their convolutions were omitted; complete count verification passed.
Default THOP omits SiLU/Sigmoid, functional add/mul, bilinear interpolation and Conv/Linear bias adds.
The expanded estimate adds these under documented assumptions; it is not exact kernel instructions or a selectively smaller profiler result.
Bilinear assumes precomputed weights, excluding coordinates; unit-cost exp is conventional, not actual GPU cost.
Counts exclude memory traffic, dispatch, preprocessing and backward; GPU wall-clock is measured separately.
Inference THOP parameters may be66 below full PyTorch total because region/distance predictors are skipped; official total retains them.
GlobalContextPath uses depthwise dilated context, not all-to-all attention.
The source fingerprint is first created here; Phase4 recorded settings/counts but not previous architecture-file hashes.
"""
(OUTPUT/"profiler_notes.txt").write_text(notes,encoding="utf-8")
top_compute=module_table[module_table.path.eq("inference")].nlargest(5,"conv_linear_macs")
top_params=parameters.nlargest(5,"parameters")
text=f"""MorphoNet-BT — Phase 5: no training.
Input: [1,1,256,256], FP32. Total={total}; Trainable={trainable}; Non-trainable={nontrainable}.
FP32: {storage['fp32_MB']:.6f} MB / {storage['fp32_MiB']:.6f} MiB.
Inference Conv/Linear MACs={profiles['inference']['independent_conv_linear_macs']}; FLOPs=2×MACs={profiles['inference']['conv_linear_flops']}.
Inference expanded arithmetic estimate={profiles['inference']['expanded_flops_estimate']} operations/FLOPs under stated assumptions.
Full auxiliary Conv/Linear MACs={profiles['auxiliary']['independent_conv_linear_macs']}; FLOPs=2×MACs={profiles['auxiliary']['conv_linear_flops']}.
Full auxiliary expanded arithmetic estimate={profiles['auxiliary']['expanded_flops_estimate']}.
THOP and independent counts agree; mixed raw counts/additional operations are in JSON and profiler_notes.txt.
Inference includes Stage2 bottleneck, boundary predictor, resize to16×16 and BAA; skips region/distance predictors and256×256 maps.
Depthwise audit: {len(depthwise)} layers PASS. No hidden256×256 convolution; manual counts agree.
Largest compute components:
{top_compute[['module','conv_linear_macs']].to_string(index=False)}
Largest parameter components:
{top_params[['module','parameters']].to_string(index=False)}
RTX3070 Laptop GPU latency:
{latency[['path','mean_ms','std_ms','median_ms','images_per_second']].to_string(index=False)}
Peak allocated MiB:
{memory[['path','batch_size','peak_allocated_MiB','incremental_forward_peak_MiB']].to_string(index=False)}
<1M parameters: PASS; <1GFLOP under stated estimate: PASS. Architecture sources/settings/splits unchanged.
The complexity target does not require architecture changes. Phase6 awaits researcher approval.
"""
(OUTPUT/"model_complexity.txt").write_text(text,encoding="utf-8")
print(text)

## Historical conclusion
Classification and auxiliary paths were measured separately without altering either. Boundary guidance is part of inference and its count. Official parameter count includes the full model. Computational scope/exclusions are explicit; memory/latency are hardware-specific. No training/baselines/ablations were performed in this phase; complexity does not establish accuracy. The historical workflow stopped before Phase 6.
**Recorded original full-model results (later A5):** 579,599 parameters; inference 275,715,968 Conv/Linear MACs, 551,431,936 FLOPs, expanded estimate 606,759,407. Auxiliary: 275,781,504 MACs; expanded estimate 608,530,927. Consistency/depthwise checks passed. Memory baseline includes retained runtime allocations and allocator rounding, not only parameters. Parameters occupy 2,318,396 bytes and buffers 45,032 bytes. Forward peak is not training or total CUDA-process memory. These are historical A5 measurements, not the principal A4 values.